# Training Synopsis Data

In [1]:
# importing data
import pandas as pd
df = pd.read_csv('../data/anime_dataset_clean.csv')

In [2]:
# seperating text and target variable
df = df[['synopsis','high_rated']]

## Text Data Preparation

In [3]:
# importing libraries
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sharm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sharm\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
# stopword cleaning and lemmatization 
corpus = []
stop_words = set(stopwords.words('english'))  # Create stopwords set only once (outside the loop)

for i in range(len(df)):
    review = re.sub('[^a-zA-Z]', ' ', df['synopsis'][i])  # Remove special characters
    review = review.lower()                               # Lowercase
    review = review.split()                               # Split into words
    
    # Stopword removal + Lemmatization
    review = [lemmatizer.lemmatize(word) for word in review if word not in stop_words]
    
    review = ' '.join(review)
    corpus.append(review)

## Model Training

In [6]:
# Independent and dependent feature
X = corpus
y =df['high_rated']

# Train Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,stratify=y,random_state=42,test_size=0.25)

In [7]:
# Importing libraries and defining models
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    "MultinomialNB": Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))),
        ('clf', MultinomialNB())
    ])
}

In [11]:
# Model Training & Evaluation
results = []
for model_name, pipeline in models.items():
    # Fit pipeline on raw text (TF-IDF vectorizer fits strictly on X_train)
    pipeline.fit(X_train, y_train)
    
    # Generate predictions directly from raw text
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    y_train_proba = pipeline.predict_proba(X_train)[:, 1]
    y_test_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # Store metrics
    results.append({
        "Model": model_name,
        "Train Macro F1": f1_score(y_train, y_train_pred, average='macro'),
        "Test Macro F1": f1_score(y_test, y_test_pred, average='macro'),
        "Train ROC-AUC": roc_auc_score(y_train, y_train_proba),
        "Test ROC-AUC": roc_auc_score(y_test, y_test_proba),
        "Test Precision (Class 1)": precision_score(y_test, y_test_pred),
        "Test Recall (Class 1)": recall_score(y_test, y_test_pred)
    })

# 4. Display Summary Table
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

              Model  Train Macro F1  Test Macro F1  Train ROC-AUC  Test ROC-AUC  Test Precision (Class 1)  Test Recall (Class 1)
Logistic Regression        0.783342       0.701236       0.894274      0.802503                  0.527162               0.718821
      MultinomialNB        0.745993       0.706174       0.826215      0.783351                  0.620782               0.528345


### Observation & Conclusion:
- Logistic Regression provide better results compare to MultinomialNB in Test ROC-AUC (`0.803 vs 0.783`) & Recall(`0.719 vs 0.528`). But MultinomialNB slightly provide better when looking at Test Macro F1(`0.706 vs 0.701`) &  Precision(`0.621 vs 0.527`)
- Logistic Regression will be better for our problem because of its high *Recall*, because missing an actual high rated anime is a worse outcome than occasionally flagging a mediocre one. So we prioritize catching as many high-rated anime as possible.

In [12]:
import joblib
import os

# Create the folder if it doesn't already exist
os.makedirs('../models', exist_ok=True)

# Save the fitted NLP pipeline
joblib.dump(models['Logistic Regression'], '../models/nlp_baseline_pipeline.pkl')

print("NLP pipeline saved successfully to '../models/nlp_baseline_pipeline.pkl'!")

NLP pipeline saved successfully to '../models/nlp_baseline_pipeline.pkl'!
